# BGG API

Exploring the BGG XML API2 for requests

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 23/03/2026   | Martin | Create  | Initial ingestion pipeline components | 
| 31/03/2026   | Martin | Update  | Completed BGG API scraper | 

# Content

* [Request & XML](#request--xml)
* [Checking DB](#checking-db)

# Request & XML

Pulling and exploring the returned XML data

In [1]:
import os
import requests
import pandas as pd
from xml.etree import ElementTree
from dotenv import load_dotenv

load_dotenv()

True

1. Get the list of games that can be pulled 

In [3]:
df_bg = pd.read_csv("../data/boardgames_ranks.csv")
df_bg.head()

,id,name,yearpublished,rank,bayesaverage,average,usersrated,is_expansion,abstracts_rank,cgs_rank,childrensgames_rank,familygames_rank,partygames_rank,strategygames_rank,thematic_rank,wargames_rank,pulled
0,224517,Brass: Birmingham,2018,1,8.39530,8.56746,57227,0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1
1,342942,Ark Nova,2021,2,8.35524,8.54188,59656,0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,1
2,161936,Pandemic Legacy: Season 1,2015,3,8.34784,8.50432,57122,0,NaN,NaN,NaN,NaN,NaN,3.0,1.0,NaN,1
3,174430,Gloomhaven,2017,4,8.30257,8.54350,66778,0,NaN,NaN,NaN,NaN,NaN,5.0,2.0,NaN,1
4,316554,Dune: Imperium,2020,5,8.22374,8.41359,57460,0,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,1


In [17]:
def get_bgg_id(df: pd.DataFrame) -> int:
  """
  Get the highest ranking ID of the board game that hasn't been pulled yet

  Args:
      df (pd.DataFrame): boardgames_ranks df

  Returns:
      int: BGG ID
  """
  filtered_df = df[df['pulled'] != 1]
  return filtered_df['id'].to_numpy()[0]

In [16]:
def update_bgg_id(df:pd.DataFrame, id: int) -> pd.DataFrame:
  """
  Update the pulled column of the dataframe based on the BGG ID

  Args:
      df (pd.DataFrame): boardgames_ranks df
      id (int): BGG ID

  Returns:
      pd.DataFrame: Updated dataframe based on BGG ID
  """
  df.loc[df['id'] == id, 'pulled'] = 1
  return df

2. Pulling and compiling data

In [21]:
def get_suggested_players(tree):
  poll = tree.find(".//poll[@name='suggested_numplayers']")
  suggested = max(
    poll.findall("results"),
    key=lambda r: int(r.find("result[@value='Best']").get("numvotes"))
  )
  return suggested.get("numplayers")

def get_suggested_age(tree):
  poll = tree.find(".//poll[@name='suggested_playerage']")
  suggested = max(
    poll.findall(".//result"),
    key=lambda r: int(r.get("numvotes"))
  )
  return suggested.get("value")

def get_link_type_list(tree, tag):
  v = [l.get('value') for l in tree.findall(f".//link[@type='{tag}']")]
  i = [int(l.get('id')) for l in tree.findall(f".//link[@type='{tag}']")]
  return i, v
  


In [ ]:
# tree.findall(".//link[@type='boardgamecategory']")

[str, str, str, str, str, str]

In [22]:
BGG_ID = 224517
TOKEN = os.environ['BGG_TOKEN']
PAGE = 1
URL = f"https://boardgamegeek.com/xmlapi2/thing?id={BGG_ID}&type=boardgame&stats=1&comments=1&page={PAGE}"

response = requests.get(
  URL,
  headers={"Authorization": f"Bearer {TOKEN}"}
)

if response.status_code == 200:
  tree = ElementTree.fromstring(response.content)
else:
  print("Failed to retrieve data")

# BGG Info
bgg_info = {}
bgg_info['id'] = BGG_ID
bgg_info['name'] = tree.find(".//name[@type='primary']").get('value')
bgg_info['description'] = tree.find(".//description").text.strip('\n')
bgg_info['year_published'] = tree.find('.//yearpublished').get('value')
## Players
bgg_info['min_players'] = int(tree.find(".//minplayers").get('value'))
bgg_info['max_players'] = int(tree.find(".//maxplayers").get('value'))
bgg_info['suggested_players'] = get_suggested_players(tree)
## Playing time
bgg_info['playing_time'] = int(tree.find(".//playingtime").get('value'))
bgg_info['min_playing_time'] = int(tree.find(".//minplaytime").get('value'))
bgg_info['max_playing_time'] = int(tree.find(".//maxplaytime").get('value'))
## Age
bgg_info['min_age'] = int(tree.find(".//minage").get('value'))
bgg_info['suggested_age'] = get_suggested_age(tree)
## Tags
bgg_info['category_ids'], bgg_info['categories'] = get_link_type_list(tree, 'boardgamecategory')
bgg_info['mechanic_ids'], bgg_info['mechanics'] = get_link_type_list(tree, 'boardgamemechanic')
bgg_info['family_ids'], bgg_info['families'] = get_link_type_list(tree, 'boardgamefamily')
bgg_info['designer_ids'], bgg_info['designers'] = get_link_type_list(tree, 'boardgamedesigner')
bgg_info['artist_ids'], bgg_info['artists'] = get_link_type_list(tree, 'boardgameartist')
## Ratings
bgg_info['num_ratings'] = int(tree.find(".//usersrated").get('value'))
bgg_info['rating'] = float(tree.find(".//average").get('value'))
bgg_info['bayes_rating'] = float(tree.find(".//bayesaverage").get('value'))
bgg_info['complexity'] = float(tree.find(".//averageweight").get('value'))

In [23]:
bgg_info

{'id': 224517,
 'name': 'Brass: Birmingham',
 'description': "Brass: Birmingham is an economic strategy game sequel to Martin Wallace's 2007 masterpiece, Brass. Brass: Birmingham tells the story of competing entrepreneurs in Birmingham during the industrial revolution between the years of 1770 and 1870.\n\nIt offers a very different story arc and experience from its predecessor. As in its predecessor, you must develop, build and establish your industries and network in an effort to exploit low or high market demands. The game is played over two halves: the canal era (years 1770-1830) and the rail era (years 1830-1870). To win the game, score the most VPs. VPs are counted at the end of each half for the canals, rails and established (flipped) industry tiles.\n\nEach round, players take turns according to the turn order track, receiving two actions to perform any of the following actions (found in the original game):\n\n1) Build - Pay required resources and place an industry tile.\n2) Ne

In [3]:
BGG_ID = 224517
TOKEN = os.environ['BGG_TOKEN']
URL = f"https://boardgamegeek.com/xmlapi2/thing?id={BGG_ID}&type=boardgame&ratingcomments=1&page=600"

response = requests.get(
  URL,
  headers={"Authorization": f"Bearer {TOKEN}"}
)

if response.status_code == 200:
  tree = ElementTree.fromstring(response.content)
else:
  print("Failed to retrieve data")


In [4]:
tree.findall(".//comment")

[]

In [ ]:
# Comments
MAX_PAGES = 10
MIN_WORDS = 20
URL = f"https://boardgamegeek.com/xmlapi2/thing?id={BGG_ID}&type=boardgame&stats=1&comments=1&page={PAGE}"

comments = []
all_comments = tree.findall(".//comment")

for comment in all_comments:
  text = comment.get('value').strip()
  if len(text.split(' ')) < MIN_WORDS:
    continue

  info = {
    'rating': comment.get('rating'),
    'comment': comment.get('value')
  }
  comments.append(info)

In [6]:
import random
import time

# Rating Comments
MAX_PAGES = 20
MIN_WORDS = 20
MIN_COMMENTS = 10
PAGE_RANGE = 550

# Variables
comments = []
page_numbers = [i for i in range(1, PAGE_RANGE+1)]
searched_pages = 1
it = 0

while searched_pages < MAX_PAGES:
  it += 1
  page = random.choice(page_numbers)
  url = f"https://boardgamegeek.com/xmlapi2/thing?id={BGG_ID}&type=boardgame&ratingcomments=1&page={page}"
  print(f"Iteration: {it} | Page: {page} | Number of Comments: {len(comments)} | Number of pages searched: {searched_pages}")

  # Retrieve page
  response = requests.get(
    url,
    headers={"Authorization": f"Bearer {TOKEN}"}
  )

  if response.status_code == 200:
    tree = ElementTree.fromstring(response.content)
  else:
    print("Failed to retrieve data")
    break
  
  # Check if there are any comments on the page
  all_comments = tree.findall(".//comment")
  if not all_comments:
    continue

  # Collect and store comments
  for comment in all_comments:
    text = comment.get('value').strip()
    if len(text.split(' ')) < MIN_WORDS:
      continue

    info = {
      'rating': comment.get('rating'),
      'comment': comment.get('value')
    }
    comments.append(info)
  
  searched_pages += 1
  
  if len(comments) > MIN_COMMENTS:
    break

  time.sleep(10)

Iteration: 1 | Page: 501 | Number of Comments: 0 | Number of pages searched: 1
Iteration: 1 | Page: 515 | Number of Comments: 5 | Number of pages searched: 2
Iteration: 1 | Page: 109 | Number of Comments: 6 | Number of pages searched: 3


In [7]:
comments

[{'rating': '10',
  'comment': "Arguably the best game I've played with three players. It's also fun with two, and I have yet to try it with four. It's not the easiest game to teach, and it takes a few rounds to understand everything, but we had no problem learning it or teaching it to others. Also, great art! This really is a masterpiece game in all areas, and it is reasonably priced compared to other games in its class."},
 {'rating': '10',
  'comment': 'Theme:9 (macro, realistic, not very interesting to me) Component:9 (Roxley product, high quality, cute barrels) Mechanics:9 (works so well to deliver the theme) Complexity:8 (simple action. complex mind game BGG3.9) Tension:9 (brain explosion in late game)  Essential! '},
 {'rating': '10',
  'comment': 'One of the best euros ever designed. Although most interesting while exploring the systems, it remains a joy to bring to the table even after numerous plays.'},
 {'rating': '10',
  'comment': 'This is a brilliant game. Even if the the

# DuckDB Testing

In [1]:
import duckdb

In [28]:
def setup_duckdb(duckdb_path: str):
  con = duckdb.connect(duckdb_path)

  # Create schema
  con.execute(f"CREATE SCHEMA IF NOT EXISTS bgg;")

  # Create table 1: games
  print("Creating games table...")
  create_games_table = f"""
CREATE TABLE IF NOT EXISTS bgg.games (
  id INTEGER PRIMARY KEY,
  name VARCHAR,
  description VARCHAR,
  year_published VARCHAR,
  min_players INTEGER,
  max_players INTEGER,
  suggested_players VARCHAR,
  playing_time INTEGER,
  min_playing_time INTEGER,
  max_playing_time INTEGER,
  min_age INTEGER,
  suggested_age VARCHAR,
  category_ids INTEGER[],
  categories VARCHAR[],
  mechanic_ids INTEGER[],
  mechanics VARCHAR[],
  family_ids INTEGER[],
  families VARCHAR[],
  designer_ids INTEGER[],
  designers VARCHAR[],
  artist_ids INTEGER[],
  artists VARCHAR[],
  num_ratings INTEGER,
  rating DOUBLE,
  bayes_rating DOUBLE,
  complexity DOUBLE
);
  """
  con.execute(create_games_table)
  print("Games table successfully created.")

  # Create table 2: comments
  print("Creating comments table...")
  con.execute("CREATE SEQUENCE seq_id START 1;")
  create_comments_table = f"""
CREATE TABLE IF NOT EXISTS bgg.comments (
  id INTEGER PRIMARY KEY DEFAULT nextval('seq_id'),
  bgg_id INTEGER,
  rating INTEGER,
  comment TEXT
);
  """
  con.execute(create_comments_table)
  print("Comments table successfully created.")

  return con

In [29]:
con = setup_duckdb("../data/test.duckdb")

Creating games table...
Games table successfully created.
Creating comments table...
Comments table successfully created.


In [32]:
con.sql("SHOW ALL TABLES")

┌──────────┬─────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │   name   │                                                                                                                                                             column_names                                                                                                                                                       

In [36]:
bgg_info_rows = {k: [v] for k, v in bgg_info.items()}
insert_df = pd.DataFrame.from_dict(bgg_info_rows)
con.execute("INSERT INTO bgg.games SELECT * FROM insert_df")

In [37]:
con.sql("SELECT * FROM bgg.games")

┌────────┬───────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [11]:
con.close()

---

# Checking DB

In [4]:
import duckdb

DB_PATH = "../data/bgg_db.duckdb"

In [5]:
con = duckdb.connect(DB_PATH)

In [6]:
con.sql("SHOW ALL TABLES;")

┌──────────┬─────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │   name   │                                                                                                                                                             column_names                                                                                                                                                       

In [7]:
con.sql("SELECT * FROM bgg.games LIMIT 10;")

┌────────┬───────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [2]:
%watermark

Last updated: 2025-06-18T19:03:45.452311+08:00

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.31.0

Compiler    : MSC v.1938 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 20
Architecture: 64bit

